## Runtime Compatibility Check

Run this notebook from the repo root or from a Kaggle working directory that contains the repo. The notebook is intentionally thin and delegates training, evaluation, and metrics work to repo modules.


In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path


def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path("/kaggle/working"), Path("/kaggle/input")]
    for candidate in candidates:
        if (candidate / "src" / "chakra").exists():
            return candidate
    return Path.cwd()


REPO_ROOT = find_repo_root()
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO_ROOT / "src")
VERSION = "vR.1"
PARENT = None
CONTROL_CONFIG = "configs/hndsr_vr/vR.1_control.yaml"
SMOKE_CONFIG = "configs/hndsr_vr/vR.1_smoke.yaml"
TRAIN_CONFIG = "configs/hndsr_vr/vR.1_train.yaml"
print(f"Repo root: {REPO_ROOT}")
print(f"Python: {sys.executable}")
print(f"Platform: {platform.platform()}")
print(f"Kaggle runtime type: {os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'local-or-unknown')}")
assert (REPO_ROOT / 'src' / 'chakra').exists(), 'Expected src/chakra under repo root.'


## Post-Restart GPU Sanity Check

Run the next cell after any runtime restart. If CUDA is not available, keep the run diagnostic-focused and record the limitation in the handoff note.


In [ ]:
import torch
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


# vR.1 HNDSR

## Scratch SR3 Kaggle Baseline

This notebook is an execution shell around repo-owned code, not a notebook-only model implementation.


## Experiment Registry

- Version: `vR.1`
- Parent: `none`
- Lineage: `scratch`
- Commands: `python -m chakra validate-version --version vR.1`, `python -m chakra.hndsr_vr.train_runner --config`, `python -m chakra.hndsr_vr.evaluate_runner --config`


## Paper Lineage and Hypothesis

The goal of `vR.1` is not to claim paper-quality output yet. The goal is to prove a traceable, repo-owned Kaggle loop that can later support deeper SR3 and HNDSR ablations.


## Dataset and Config Contract

- Control config: `configs/hndsr_vr/vR.1_control.yaml`
- Smoke config: `configs/hndsr_vr/vR.1_smoke.yaml`
- Train config: `configs/hndsr_vr/vR.1_train.yaml`
- Fixed lane: Kaggle paired `4x` data


## Weights & Biases Setup

W&B online logging is required for milestone completion. Configure `WANDB_API_KEY` in Kaggle Secrets before running the train/eval cells.


## Training Execution

Run the validation cell first, then the control evaluation, then smoke train/eval, then the full train/eval path.


In [ ]:
subprocess.run([sys.executable, '-m', 'chakra', 'validate-version', '--version', 'vR.1'], cwd=REPO_ROOT, env=ENV, check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "chakra.hndsr_vr.evaluate_runner", "--config", CONTROL_CONFIG, "--run-name", f"{VERSION}-control"], cwd=REPO_ROOT, env=ENV, check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "chakra.hndsr_vr.train_runner", "--config", SMOKE_CONFIG, "--run-name", f"{VERSION}-smoke"], cwd=REPO_ROOT, env=ENV, check=True)
subprocess.run([sys.executable, "-m", "chakra.hndsr_vr.evaluate_runner", "--config", SMOKE_CONFIG, "--run-name", f"{VERSION}-smoke-eval", "--checkpoint", f"artifacts/{VERSION}-smoke/checkpoints/vR.1_smoke_best.pt"], cwd=REPO_ROOT, env=ENV, check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "chakra", "validate-version", "--version", VERSION], cwd=REPO_ROOT, env=ENV, check=True)
subprocess.run([sys.executable, "-m", "chakra.hndsr_vr.train_runner", "--config", TRAIN_CONFIG, "--run-name", f"{VERSION}-train"], cwd=REPO_ROOT, env=ENV, check=True)
subprocess.run([sys.executable, "-m", "chakra.hndsr_vr.evaluate_runner", "--config", TRAIN_CONFIG, "--run-name", f"{VERSION}-eval", "--checkpoint", f"artifacts/{VERSION}-train/checkpoints/vR.1_train_best.pt"], cwd=REPO_ROOT, env=ENV, check=True)

## Evaluation and Export

The evaluation runner writes JSON summaries, sample grids, and tracker metadata under `artifacts/`. Pull the Kaggle bundle back before review.


## Results Dashboard

Use the next cell to inspect any evaluation summaries already produced by the run.


In [ ]:
from pathlib import Path
import json

metrics_root = REPO_ROOT / "artifacts"
for candidate in [
    metrics_root / f"{VERSION}-eval" / "metrics" / "eval_summary.json",
    metrics_root / f"{VERSION}-smoke-eval" / "metrics" / "eval_summary.json",
    metrics_root / f"{VERSION}-control" / "metrics" / "eval_summary.json",
]:
    if candidate.exists():
        print(candidate)
        print(json.dumps(json.loads(candidate.read_text(encoding="utf-8")), indent=2))


## Troubleshooting and Known Failure Modes

- Missing W&B auth means the run is not milestone-complete.
- CPU-only Kaggle runs are valid for diagnostics, not promotion.
- Any Kaggle-side fix must be mirrored back into repo code or docs before the version is frozen.


## Changelog

- Initial scaffold for the immutable Kaggle baseline.


## Next Step Gate

Freeze only after outputs are pulled, synced, reviewed, and mirrored. Fork the next version only from a written promotion decision.
